# AEGIS — Day 2 | Forecasting, RAG and managed streaming

**Prerequisite:** Run and verify Day 1 on your own GCP project. This notebook can be opened in a fresh Colab runtime and will clone your GitHub repository automatically.

**Cost gates:** BigQuery ML and managed Dataflow may consume trial credits. Vertex AI/Gemini usage requires separate billing eligibility checks. The local Decision Replay and retrieval baseline do not need GCP calls.


## 1. Load the repository (works in a fresh Colab runtime)

This step uses the same GitHub source as Day 1. Run the next cell to fetch the code and install lightweight dependencies. We do **not** republish streaming events automatically.


In [ ]:
import os
import subprocess
from pathlib import Path

project_dir = Path('/content/aegis-real-time-supply-chain-ai')
repo_url = 'https://github.com/Codewithadi21/aegis-real-time-supply-chain-ai.git'
if not (project_dir / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(project_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only'], check=True)
os.chdir(project_dir)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project is ready:', os.getcwd())
# For optional GCP exercises, install requirements-gcp.txt and authenticate
# via google.colab.auth.authenticate_user(), then set PROJECT_ID manually.


## 2. Calculate supplier delay scenarios (no cloud charges)

In [ ]:
import pandas as pd
from aegis.risk import decision_replay
scenarios = pd.DataFrame(decision_replay())
display(scenarios)

## 3. BigQuery ML (optional, may use credits)

Open `sql/03_bigquery_ml_optional.sql` in [BigQuery Studio](https://console.cloud.google.com/bigquery). Replace `{{PROJECT_ID}}` with your real ID. Run the `CREATE MODEL` statement first, then the evaluation statement, then baseline MAE. Do not report accuracy until you've actually run it.

## 4. Grounded supplier policy retrieval (free baseline)

This baseline uses TF-IDF vectors. It is not a neural embedding model; it shows the retrieval-and-grounding workflow.

In [ ]:
from aegis.retrieval import search_policies, draft_evidence_prompt
question = 'What escalation is required for a supplier shipment delayed seven days?'
found = search_policies(question)
display(pd.DataFrame([{'document':d['source'],'similarity':d['score']} for d in found]))
prompt = draft_evidence_prompt(question,found,decision_replay())
print(prompt[:1600])

## 5. Neural embeddings (optional, free Colab CPU; model download required)

In [ ]:
# Uncomment if you want to download an embedding model and perform semantic vector retrieval:
# %pip install -q -r requirements-rag-optional.txt
# from aegis.semantic_retrieval import semantic_search
# semantic_docs = semantic_search(question)
# print([(d['source'],round(d['score'],3)) for d in semantic_docs])

## 6. Gemini via Vertex AI (optional paid service)

Only after checking that Vertex AI billing, region and model access are available:

```python
from aegis.retrieval import ask_vertex_gemini
answer = ask_vertex_gemini(prompt, PROJECT_ID)
print(answer)
```

This API call is deliberately NOT run by default. Model pricing and access can change.

## 7. Managed Dataflow (optional, credit-consuming)

Read `docs/DAY2_ADVANCED.md` carefully. For Dataflow, create a *separate pull subscription* and a GCS staging bucket, configure IAM, and run the Beam pipeline only while actively testing. Drain the Dataflow job in the Google Console after capturing proof.

Do not claim a managed Dataflow deployment until the dashboard shows an actual running/terminated job and your output tables contain events.